In [2]:
#!/usr/bin/env python3

from pathlib import Path
import numpy as np
import xarray as xr

ERA5_ROOT = Path("../data/era5")
OUT_ROOT = Path("../data/era5_derived")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

START_YEAR = 1980
END_YEAR = 2024

U_DIR = ERA5_ROOT / "u_component_of_wind"
V_DIR = ERA5_ROOT / "v_component_of_wind"
Q_DIR = ERA5_ROOT / "specific_humidity"

WIND_OUT = OUT_ROOT / "wind_speed_925"
MFX_U_OUT = OUT_ROOT / "moisture_flux_u_925"
MFX_V_OUT = OUT_ROOT / "moisture_flux_v_925"
MFX_MAG_OUT = OUT_ROOT / "moisture_flux_mag_925"

for d in [WIND_OUT, MFX_U_OUT, MFX_V_OUT, MFX_MAG_OUT]:
    d.mkdir(parents=True, exist_ok=True)


def monthly_file(var_dir: Path, stem: str, level: int, year: int, month: int) -> Path:
    return var_dir / f"{stem}_{level}_{year}_{month:02d}.nc"


def open_month(year: int, month: int):
    fu = monthly_file(U_DIR, "u_component_of_wind", 925, year, month)
    fv = monthly_file(V_DIR, "v_component_of_wind", 925, year, month)
    fq = monthly_file(Q_DIR, "specific_humidity", 925, year, month)

    if not (fu.exists() and fv.exists() and fq.exists()):
        missing = [str(p) for p in [fu, fv, fq] if not p.exists()]
        raise FileNotFoundError(f"Missing files for {year}-{month:02d}: {missing}")

    du = xr.open_dataset(fu)
    dv = xr.open_dataset(fv)
    dq = xr.open_dataset(fq)

    u = du["u_component_of_wind"]
    v = dv["v_component_of_wind"]
    q = dq["specific_humidity"]

    u, v, q = xr.align(u, v, q, join="exact")
    return u, v, q


for year in range(START_YEAR, END_YEAR + 1):
    for month in range(1, 13):
        print(f"Processing derived fields for {year}-{month:02d}")

        try:
            u, v, q = open_month(year, month)

            wind_speed = np.sqrt(u**2 + v**2).rename("wind_speed_925")
            mfx_u = (q * u).rename("moisture_flux_u_925")
            mfx_v = (q * v).rename("moisture_flux_v_925")
            mfx_mag = np.sqrt((q * u)**2 + (q * v)**2).rename("moisture_flux_mag_925")

            enc = {
                wind_speed.name: {"zlib": True, "complevel": 4},
                mfx_u.name: {"zlib": True, "complevel": 4},
                mfx_v.name: {"zlib": True, "complevel": 4},
                mfx_mag.name: {"zlib": True, "complevel": 4},
            }

            wind_speed.to_netcdf(
                WIND_OUT / f"wind_speed_925_{year}_{month:02d}.nc",
                encoding={wind_speed.name: enc[wind_speed.name]}
            )
            mfx_u.to_netcdf(
                MFX_U_OUT / f"moisture_flux_u_925_{year}_{month:02d}.nc",
                encoding={mfx_u.name: enc[mfx_u.name]}
            )
            mfx_v.to_netcdf(
                MFX_V_OUT / f"moisture_flux_v_925_{year}_{month:02d}.nc",
                encoding={mfx_v.name: enc[mfx_v.name]}
            )
            mfx_mag.to_netcdf(
                MFX_MAG_OUT / f"moisture_flux_mag_925_{year}_{month:02d}.nc",
                encoding={mfx_mag.name: enc[mfx_mag.name]}
            )

        except Exception as e:
            print(f"Failed {year}-{month:02d}: {e}")

Processing derived fields for 1980-01
Processing derived fields for 1980-02
Processing derived fields for 1980-03
Processing derived fields for 1980-04
Processing derived fields for 1980-05
Processing derived fields for 1980-06
Processing derived fields for 1980-07
Processing derived fields for 1980-08
Processing derived fields for 1980-09
Processing derived fields for 1980-10
Processing derived fields for 1980-11
Processing derived fields for 1980-12
Processing derived fields for 1981-01
Processing derived fields for 1981-02
Processing derived fields for 1981-03
Processing derived fields for 1981-04
Processing derived fields for 1981-05
Processing derived fields for 1981-06
Processing derived fields for 1981-07
Processing derived fields for 1981-08
Processing derived fields for 1981-09
Processing derived fields for 1981-10
Processing derived fields for 1981-11
Processing derived fields for 1981-12
Processing derived fields for 1982-01
Processing derived fields for 1982-02
Processing d